In [ ]:
import polars as pl
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
train_set = pl.scan_parquet('data/train_set.parquet')
valid_set = pl.scan_parquet('data/valid_set.parquet')
test_set = pl.scan_parquet('data/test_set.parquet')

In [ ]:
from datetime import datetime


LABEL_START = datetime(2025,10,1)
LABEL_END = datetime(2025,10,31)


In [ ]:
label_set = (
    train_set.filter(
        (pl.col('updated_date') >= pl.lit(LABEL_START)) &
        (pl.col('updated_date') <= pl.lit(LABEL_END))
    )
)

new_train_set = (
    train_set.filter(
        pl.col('updated_date') < pl.lit(LABEL_START)
    )
)

display(new_train_set.head(5).collect())
display(label_set.head(5).collect())




customer_id,item_id,price,location,discount,bill_id,quantity,event_type,updated_date
i32,str,f32,i32,f32,i32,i32,str,datetime[μs]
6291331,"""6767000000002""",285000.0,1053,0.0,137262467,1,"""Purchase""",2025-04-12 12:52:41.910
5505362,"""2265000000027""",195000.0,468,20000.0,138193104,1,"""Purchase""",2025-04-24 14:57:00.773
7694873,"""6497000000004""",312550.0,363,16450.0,138230425,1,"""Purchase""",2025-04-24 20:35:40.027
5837076,"""6587000000001""",149000.0,48,16000.0,138235471,1,"""Purchase""",2025-04-24 21:12:32.070
6781756,"""3880000000002""",79000.0,634,0.0,138239415,1,"""Purchase""",2025-04-25 08:00:46.293


customer_id,item_id,price,location,discount,bill_id,quantity,event_type,updated_date
i32,str,f32,i32,f32,i32,i32,str,datetime[μs]
4513878,"""0011000000054""",179000.0,767,120000.0,148611820,1,"""Purchase""",2025-10-04 08:47:30.810
4513878,"""3494000000057""",22500.0,767,22500.0,148611820,1,"""Purchase""",2025-10-04 08:47:30.810
9054360,"""0020130000004""",188635.0,725,10365.0,148480743,1,"""Purchase""",2025-10-02 15:21:44.647
9054360,"""5950000000001""",104270.0,725,5730.0,148480743,1,"""Purchase""",2025-10-02 15:21:44.647
9095746,"""2507000000003""",424080.0,712,40920.0,148498422,1,"""Purchase""",2025-10-03 14:53:52.180


### Feature engineering

- Tính tổng số lượng view, a2c, purchase trên tập train


In [ ]:
from datetime import datetime

cutoff_time = datetime(2025,10,1)


# Create Label
positive_labels = (
    label_set
    .group_by(['customer_id', 'item_id'])
    .agg(
        (pl.col('event_type') == 'Purchase').any().alias('label').cast(pl.Int8)
    )
)

item_features = (
    new_train_set
    .group_by('item_id')
    .agg(
        pl.len().log1p().alias('item_popularity')
    )
)



# display(popularity.collect(engine = 'streaming'))



train_features = (
    new_train_set
    .group_by(["customer_id", "item_id"])
    .agg(
        (pl.col("event_type") == "view_item").sum().alias("ui_total_view"),
        (pl.col("event_type") == "add_to_cart").sum().alias("ui_total_a2c"),
        (pl.col("event_type") == "Purchase").sum().alias("ui_total_purchase"),
        (pl.col("price").sum().log1p().alias("ui_sum_price")),
        (pl.col("quantity").sum().log1p().alias("ui_sum_quantity")),
        (pl.col("price").mean().log1p().alias("ui_mean_price")),
        (pl.col("quantity").mean().log1p().alias(("ui_mean_quantity"))),
        (pl.lit(cutoff_time) - pl.col('updated_date').max()).dt.total_days().alias('ui_days_since_last_event'),
        (pl.col('updated_date').max() - pl.col('updated_date').min()).dt.total_days().log1p().alias('ui_active_span_days')
    )
)


all_candidates = (
    pl.concat([
        train_features.select(["customer_id", "item_id"]),
        positive_labels.select(["customer_id", "item_id"]),
    ])
    .unique()
)


train_table = (
    all_candidates
    .join(
        train_features,
        on=["customer_id", "item_id"],
        how="left"
    )
    .join(
        positive_labels,
        on=["customer_id", "item_id"],
        how="left"
    )
    .join(
        item_features,
        on=['item_id'],
        how='left'
    )
    .with_columns(
        pl.col("label").fill_null(0).cast(pl.Int8),
        

        ### USER-ITEM LEVEL
        pl.col("ui_total_view").fill_null(0),
        pl.col("ui_total_a2c").fill_null(0),
        pl.col("ui_total_purchase").fill_null(0),
        pl.col("ui_sum_price").fill_null(0),
        pl.col("ui_sum_quantity").fill_null(0),
        pl.col("ui_mean_price").fill_null(0),
        pl.col("ui_mean_quantity").fill_null(0),

        pl.col("customer_id"),
        pl.col("item_id").cast(pl.String),

        (pl.when(pl.col('ui_total_a2c') > 0)
        .then(pl.col('ui_total_purchase') / pl.col('ui_total_a2c'))
        .otherwise(0)
        ).alias('ui_purchase_a2c_rate'),

        (pl.when(pl.col('ui_total_view') > 0)
        .then(pl.col('ui_total_a2c') / pl.col('ui_total_view'))
        .otherwise(0)
        ).alias('ui_a2c_view_rate'),

        (pl.col('ui_days_since_last_event').fill_null(9999).clip(0,365)
        ),
        (pl.col("ui_active_span_days").fill_null(0)
        )
    )
)

display(train_table.filter(pl.col('label') == 1).head(5).collect(engine="streaming"))

customer_id,item_id,ui_total_view,ui_total_a2c,ui_total_purchase,ui_sum_price,ui_sum_quantity,ui_mean_price,ui_mean_quantity,ui_days_since_last_event,ui_active_span_days,label,item_popularity,ui_purchase_a2c_rate,ui_a2c_view_rate
i32,str,u32,u32,u32,f32,f64,f32,f64,i64,f64,i8,f64,f64,f64
3167669,"""5607000000001""",0,0,5,11.710079,2.484907,10.100674,1.163151,33,4.174387,1,10.208543,0.0,0.0
4830203,"""6501000000007""",0,0,2,10.98531,1.098612,10.292179,0.693147,3,4.919981,1,11.12998,0.0,0.0
9262992,"""0020020000201""",0,0,1,10.239996,0.693147,10.239996,0.693147,0,0.0,1,10.879574,0.0,0.0
6887409,"""4600000000001""",0,0,11,15.887298,2.890372,13.489404,0.934309,8,4.89784,1,11.009853,0.0,0.0
7312593,"""6501000000007""",0,0,2,11.515931,1.098612,10.822794,0.693147,41,4.644391,1,11.12998,0.0,0.0


In [ ]:
# display(train_table.select(
#     pl.col("ui_sum_price").min().alias("price_min"),
#     pl.col("ui_sum_price").max().alias("price_max"),
#     pl.col("ui_sum_price").mean().alias("price_mean"),
#     pl.col("ui_sum_price").median().alias("price_median"),
#     pl.col("ui_sum_price").std().alias("price_std"),
#     pl.col("ui_sum_price").skew().alias("price_skew"),
# ).collect(engine="streaming"))

# display(train_table.select(
#     pl.col("ui_sum_quantity").min().alias("quantity_min"),
#     pl.col("ui_sum_quantity").max().alias("quantity_max"),
#     pl.col("ui_sum_quantity").mean().alias("quantity_mean"),
#     pl.col("ui_sum_quantity").median().alias("quantity_median"),
#     pl.col("ui_sum_quantity").std().alias("quantity_std"),
#     pl.col("ui_sum_quantity").skew().alias("quantity_skew"),
# ).collect(engine="streaming"))


# display(
#     train_table.select(
#         (pl.col('ui_sum_price') == 0).sum().alias('total 0 price'),
#         (pl.len().alias('total'))
#     ).collect(engine = 'streaming')
# )


# display(train_table.select(
#     pl.col("ui_days_since_last_event").min().alias("days_min"),
#     pl.col("ui_days_since_last_event").max().alias("days_max"),
#     pl.col("ui_days_since_last_event").mean().alias("days_mean"),
#     pl.col("ui_days_since_last_event").median().alias("days_median"),
#     pl.col("ui_days_since_last_event").std().alias("days_std"),
#     pl.col("ui_days_since_last_event").skew().alias("days_skew"),
#     pl.col("ui_days_since_last_event").null_count().alias("null_count"),
# ).collect(engine="streaming"))


# display(train_table.select(
#     pl.col("ui_active_span_days").min().alias("days_min"),
#     pl.col("ui_active_span_days").max().alias("days_max"),
#     pl.col("ui_active_span_days").mean().alias("days_mean"),
#     pl.col("ui_active_span_days").median().alias("days_median"),
#     pl.col("ui_active_span_days").std().alias("days_std"),
#     pl.col("ui_active_span_days").skew().alias("days_skew"),
#     pl.col("ui_active_span_days").null_count().alias("null_count"),
# ).collect(engine="streaming"))




### Training

In [ ]:
pdf = train_table.collect(engine = 'streaming').to_pandas()

X_train = pdf.drop(columns = ['label'])
y_train = pdf['label']


In [ ]:
cat_cols = ["customer_id", "item_id"]

for col in cat_cols:
    X_train[col] = X_train[col].astype("category")

In [ ]:
from lightgbm import LGBMClassifier

model = LGBMClassifier(
    objective="binary",
    n_estimators=500,
    learning_rate=0.03,
    num_leaves=31,
    max_depth=7,
    min_child_samples=100,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.0,
    reg_lambda=1.0,
    class_weight="balanced",
    random_state=42,
    verbosity=-1,
    # device_type="cuda",
    # max_bin="63"

)


In [ ]:
model.fit(
    X_train,
    y_train,
    categorical_feature=cat_cols
)

In [ ]:
import joblib

joblib.dump(model, "lightgbm_model_1.pkl")

['lightgbm_model_1.pkl']

### Evaluation

In [ ]:
import joblib
from lightgbm import LGBMClassifier
#model = LGBMClassifier(
#    objective="binary",
#    n_estimators=300,
#    learning_rate=0.05,
#    num_leaves=31,
#    class_weight="balanced",
#    random_state=42,
#)

model = joblib.load('lightgbm_model_1.pkl')


In [ ]:
ground_truth = (
    valid_set.filter(
        pl.col('event_type') == 'Purchase'
    )
    .select(['customer_id', 'item_id'])
    .unique()
)

display(ground_truth.collect(engine='streaming'))

customer_id,item_id
i32,str
9431962,"""3494000000066"""
9367280,"""6678000000007"""
5282649,"""7410000000002"""
3866228,"""3389000000294"""
8477924,"""2506000000003"""
…,…
5111221,"""1308000000002"""
590842,"""4603024000002"""
2436313,"""6697000000004"""


In [ ]:
purchase_valid_set = valid_set.filter(pl.col('event_type') == 'Purchase')
user_list = (purchase_valid_set.select('customer_id').unique().sort('customer_id'))

item_list = (new_train_set.select(['item_id']).unique().sort('item_id'))



In [ ]:
user_list_head = user_list.head(500)

valid_table = user_list_head.join(
    item_list, how = 'cross'
)

# display(user_list_head.collect(engine = 'streaming'))

In [ ]:
X_table = X_train

X_table['score'] = model.predict_proba(X_table)[:,1]

X_table = pl.from_pandas(X_table)

X_table.write_parquet('train_table_1.parquet')

In [ ]:
X_table = pl.scan_parquet('train_table_1.parquet')

X_table = X_table.with_columns(pl.col('item_id').cast(pl.String))



In [ ]:
results = (
    valid_table.select(['customer_id', 'item_id']).join(
        X_table, on = ['customer_id', 'item_id'], how='left'
    )
    .with_columns(
        pl.col('score').fill_null(0)
    )
    .select(['customer_id', 'item_id', 'score'])
)


results.sink_parquet('result_table.parquet')

results = results.collect(engine = 'streaming')


In [ ]:

hits = 0
for x in (user_list_head.collect(engine = 'streaming').to_series()):
    # print(x)
    pred = results.filter(pl.col('customer_id') == x).sort('score', descending=True).select('item_id').head(10).to_series().to_list()
    gt = ground_truth.filter(pl.col('customer_id') == x).select('item_id').collect(engine = 'streaming').to_series().to_list()
    cnt = len(set(pred) & set(gt))
    hits += cnt / 10
    # print(cnt)

print(hits/5)


9.620000000000019
